Imports

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os

In [2]:
street_clean_df = pd.read_csv("street_clean.csv")
outcome_clean_df = pd.read_csv("outcome_clean.csv")
merged_df = pd.read_csv("merged_clean.csv")
pop_df = pd.read_csv("population_clean.csv")

Checking the .shape of the cleaned street data, cleaned outcome data, merged dataset, and population data provides a quick validation that each file has loaded correctly and that the cleaning steps have produced the expected number of rows and columns. Comparing these shapes helps confirm that no data was accidentally lost, duplicated, or mis‑aligned during import and preprocessing. as well as various other checks. 


In [3]:
street_clean_df.shape, outcome_clean_df.shape, merged_df.shape,pop_df.shape

((3040737, 13), (2617845, 12), (3040737, 14), (172, 352))

In [4]:
pop_df.isna().sum().sum()

np.int64(0)

In [5]:
merged_df["missing_location"] = merged_df["latitude"].isna()
merged_df.head(2)

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,year,month_num,outcome_type,missing_location
0,000004a8bc59a7595f94d35c8b02633eb0dd8077305e28...,2023-07,Lancashire Constabulary,Lancashire Constabulary,-2.503001,53.752564,On or near Leamington Road,E01012588,Blackburn with Darwen 002E,Violence and sexual offences,Status update unavailable,2023,7,NaN,False
1,0000067195b659bd3f0b70409a59cd11816770401ea366...,2025-12,Cambridgeshire Constabulary,Cambridgeshire Constabulary,0.113614,52.173247,On or near Anstey Way,E01035526,Cambridge 015C,Criminal damage and arson,Investigation complete; no suspect identified,2025,12,Investigation complete; no suspect identified,False


In [6]:
merged_df["has_outcome"] = merged_df["outcome_type"].notna()
merged_df["has_outcome"].value_counts()

has_outcome
True     2615091
False     425646
Name: count, dtype: int64

In [7]:
merged_df["crime_type"].value_counts()

crime_type
Violence and sexual offences    1379915
Criminal damage and arson        289031
Public order                     282387
Shoplifting                      278180
Other theft                      238365
Vehicle crime                    156454
Burglary                         131547
Drugs                             90199
Other crime                       79071
Possession of weapons             34886
Bicycle theft                     33652
Robbery                           27213
Theft from the person             19837
Name: count, dtype: int64

This mapping groups the detailed police crime types into broader, more interpretable categories. The raw crime_type field contains many specific labels, some of which overlap or represent very similar behaviours. By creating crime_category_map and applying it with .map(), each crime is assigned to a consistent high‑level category such as Violence, Theft, Vehicle Crime, or Damage. This simplifies the dataset, reduces noise, and makes it easier to analyse trends, compare forces, and produce clearer visualisations without losing the essential meaning of each crime type.

In [8]:
crime_category_map = {
    # Violence & Harm
    "Violence and sexual offences": "Violence and sexual offences",
    "Public order": "Public order",
    "Possession of weapons": "Possession of weapons",

    # Theft & Property Crime
    "Burglary": "Theft",
    "Robbery": "Theft",
    "Shoplifting": "Theft",
    "Bicycle theft": "Theft",
    "Other theft": "Theft",
    "Theft from the person": "Theft",

    # Vehicle Crime
    "Vehicle crime": "Vehicle Crime",

    # Criminal Damage
    "Criminal damage and arson": "Damage",

    # Drugs
    "Drugs": "Drugs",

    # Miscellaneous
    "Other crime": "Other",
}
merged_df["crime_category"] = merged_df["crime_type"].map(crime_category_map)


This mapping assigns each police force area to a broader urbanisation category based on the general character of the places it covers. The raw falls_within field lists the police force name, but this alone doesn’t indicate whether the area is predominantly urban, rural, or mixed. By creating urban_map and applying it with .map(), each crime record is given an urbanisation label such as Urban City & Town, Rural Town & Fringe, or Urban Minor Conurbation. This allows for meaningful comparisons of crime patterns across different types of areas and supports analysis of how urbanisation influences crime levels, trends, and types.

In [9]:
urban_map = {
    "Avon and Somerset Constabulary": "Urban City & Town",
    "Cambridgeshire Constabulary": "Urban City & Town",
    "Cheshire Constabulary": "Urban City & Town",
    "Cumbria Constabulary": "Rural Village & Dispersed",
    "Derbyshire Constabulary": "Urban City & Town",
    "Durham Constabulary": "Urban City & Town",
    "Gloucestershire Constabulary": "Rural Town & Fringe",
    "Hampshire Constabulary": "Urban City & Town",
    "Hertfordshire Constabulary": "Urban City & Town",
    "Lancashire Constabulary": "Urban Minor Conurbation",
    "Norfolk Constabulary": "Rural Town & Fringe",
    "Suffolk Constabulary": "Rural Town & Fringe"
}

merged_df["urbanisation"] = merged_df["falls_within"].map(urban_map)


merged_df.head(2)

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,year,month_num,outcome_type,missing_location,has_outcome,crime_category,urbanisation
0,000004a8bc59a7595f94d35c8b02633eb0dd8077305e28...,2023-07,Lancashire Constabulary,Lancashire Constabulary,-2.503001,53.752564,On or near Leamington Road,E01012588,Blackburn with Darwen 002E,Violence and sexual offences,Status update unavailable,2023,7,NaN,False,False,Violence and sexual offences,Urban Minor Conurbation
1,0000067195b659bd3f0b70409a59cd11816770401ea366...,2025-12,Cambridgeshire Constabulary,Cambridgeshire Constabulary,0.113614,52.173247,On or near Anstey Way,E01035526,Cambridge 015C,Criminal damage and arson,Investigation complete; no suspect identified,2025,12,Investigation complete; no suspect identified,False,True,Damage,Urban City & Town


This step compares the police force names used in the crime dataset with those used in the population dataset to ensure they refer to the same geographic areas. The population file uses slightly different naming conventions, so the pop_to_constabulary dictionary standardises these names to match the falls_within values in the crime data. After creating a consistent falls_within_constabulary column, the population dataset is filtered to include only the forces that actually appear in the merged crime data. This alignment prevents mismatches during later joins and ensures that population figures are only used for areas where crime data exists, which is essential for producing accurate per‑capita crime rates.This allows for the merging of the pop and crime data which will allow for the caluclation of crime rates.

In [10]:
merged_df["falls_within"].unique()

array(['Lancashire Constabulary', 'Cambridgeshire Constabulary',
       'Norfolk Constabulary', 'Hampshire Constabulary',
       'Avon and Somerset Constabulary', 'Derbyshire Constabulary',
       'Cheshire Constabulary', 'Suffolk Constabulary',
       'Gloucestershire Constabulary', 'Durham Constabulary',
       'Cumbria Constabulary', 'Hertfordshire Constabulary'], dtype=object)

In [11]:
merged_df["reported_by"].unique()

array(['Lancashire Constabulary', 'Cambridgeshire Constabulary',
       'Norfolk Constabulary', 'Hampshire Constabulary',
       'Avon and Somerset Constabulary', 'Derbyshire Constabulary',
       'Cheshire Constabulary', 'Suffolk Constabulary',
       'Gloucestershire Constabulary', 'Durham Constabulary',
       'Cumbria Constabulary', 'Hertfordshire Constabulary'], dtype=object)

In [12]:
pop_df["PFA 2023 Name"].unique()

array(['Metropolitan Police', 'Cumbria', 'Lancashire', 'Merseyside',
       'Greater Manchester', 'Cheshire', 'Northumbria', 'Durham',
       'North Yorkshire', 'West Yorkshire', 'South Yorkshire',
       'Humberside', 'Cleveland', 'West Midlands', 'Staffordshire',
       'West Mercia', 'Warwickshire', 'Derbyshire', 'Nottinghamshire',
       'Lincolnshire', 'Leicestershire', 'Northamptonshire',
       'Cambridgeshire', 'Norfolk', 'Suffolk', 'Bedfordshire',
       'Hertfordshire', 'Essex', 'Thames Valley', 'Hampshire', 'Surrey',
       'Kent', 'Sussex', 'London, City of', 'Devon & Cornwall',
       'Avon and Somerset', 'Gloucestershire', 'Wiltshire', 'Dorset',
       'North Wales', 'Gwent', 'South Wales', 'Dyfed-Powys'], dtype=object)

In [13]:
keep_forces = merged_df["falls_within"].unique().tolist()

pop_to_constabulary = {
    "Lancashire": "Lancashire Constabulary",
    "Cambridgeshire": "Cambridgeshire Constabulary",
    "Norfolk": "Norfolk Constabulary",
    "Hampshire": "Hampshire Constabulary",
    "Avon and Somerset": "Avon and Somerset Constabulary",
    "Derbyshire": "Derbyshire Constabulary",
    "Cheshire": "Cheshire Constabulary",
    "Suffolk": "Suffolk Constabulary",
    "Gloucestershire": "Gloucestershire Constabulary",
    "Durham": "Durham Constabulary",
    "Cumbria": "Cumbria Constabulary",
    "Hertfordshire": "Hertfordshire Constabulary"
}

pop_df["falls_within_constabulary"] = pop_df["PFA 2023 Name"].replace(pop_to_constabulary)

pop_df = pop_df[pop_df["falls_within_constabulary"].isin(keep_forces)]


In [14]:
pop_df["PFA 2023 Name"].unique()

array(['Cumbria', 'Lancashire', 'Cheshire', 'Durham', 'Derbyshire',
       'Cambridgeshire', 'Norfolk', 'Suffolk', 'Hertfordshire',
       'Hampshire', 'Avon and Somerset', 'Gloucestershire'], dtype=object)

In [15]:
crime_by_force = merged_df.groupby("falls_within")["crime_id"].count()
crime_by_force

falls_within
Avon and Somerset Constabulary    486674
Cambridgeshire Constabulary       212324
Cheshire Constabulary             228980
Cumbria Constabulary              109537
Derbyshire Constabulary           261048
Durham Constabulary               186424
Gloucestershire Constabulary      146894
Hampshire Constabulary            457719
Hertfordshire Constabulary        246389
Lancashire Constabulary           389106
Norfolk Constabulary              178922
Suffolk Constabulary              136720
Name: crime_id, dtype: int64

In [16]:
crime_by_category = merged_df["crime_category"].value_counts()
crime_by_category

crime_category
Violence and sexual offences    1379915
Theft                            728794
Damage                           289031
Public order                     282387
Vehicle Crime                    156454
Drugs                             90199
Other                             79071
Possession of weapons             34886
Name: count, dtype: int64

In [17]:
crime_by_month = merged_df.groupby(["year", "month_num"])["crime_id"].count()
crime_by_month

year  month_num
2023  2            81153
      3            88898
      4            84759
      5            88011
      6            88231
      7            87548
      8            84569
      9            84424
      10           85210
      11           80416
      12           78574
2024  1            79788
      2            77992
      3            84671
      4            83788
      5            89464
      6            88915
      7            91591
      8            89128
      9            84765
      10           87739
      11           82819
      12           79019
2025  1            79174
      2            74303
      3            86647
      4            83015
      5            88684
      6            87123
      7            91415
      8            88836
      9            82542
      10           85211
      11           82355
      12           80172
2026  1            79788
Name: crime_id, dtype: int64

In [18]:
outcome_dist = merged_df["outcome_type"].value_counts(dropna=False)
outcome_dist

outcome_type
Unable to prosecute suspect                            1238995
Investigation complete; no suspect identified           848068
NaN                                                     425646
Suspect charged                                         251615
Local resolution                                        115580
Action to be taken by another organisation               50367
Further investigation is not in the public interest      34264
Offender given a caution                                 33247
Formal action is not in the public interest              22694
Further action is not in the public interest             13700
Suspect charged as part of another case                   4927
Offender given penalty notice                             1395
Offender given a drugs possession warning                  239
Name: count, dtype: int64

In [19]:
merged_df.columns




Index(['crime_id', 'month', 'reported_by', 'falls_within', 'longitude',
       'latitude', 'location', 'lsoa_code', 'lsoa_name', 'crime_type',
       'last_outcome_category', 'year', 'month_num', 'outcome_type',
       'missing_location', 'has_outcome', 'crime_category', 'urbanisation'],
      dtype='object')

In [20]:
force_map = {
    "Avon and Somerset": "Avon and Somerset Constabulary",
    "Cambridgeshire": "Cambridgeshire Constabulary",
    "Cheshire": "Cheshire Constabulary",
    "Cumbria": "Cumbria Constabulary",
    "Derbyshire": "Derbyshire Constabulary",
    "Durham": "Durham Constabulary",
    "Gloucestershire": "Gloucestershire Constabulary",
    "Hampshire": "Hampshire Constabulary",
    "Hertfordshire": "Hertfordshire Constabulary",
    "Lancashire": "Lancashire Constabulary",
    "Norfolk": "Norfolk Constabulary",
    "Suffolk": "Suffolk Constabulary",
}

pop_df["force_clean"] = pop_df["PFA 2023 Name"].map(force_map)


In [21]:
pop_df.head()

,PFA 2023 Code,PFA 2023 Name,Year,F0,F1,F2,F3,F4,F5,F6,...,M80_weighted,M81_weighted,M82_weighted,M83_weighted,M84_weighted,M85_weighted,total_weighted_age,average_age,falls_within_constabulary,force_clean
4,E23000002,Cumbria,2021,2069.0,2089.0,2198.0,2182.0,2384.0,2432.0,2374.0,...,138640.0,134460.0,123738.0,115536.0,105588.0,498440.0,22380241.0,44.695316,Cumbria Constabulary,Cumbria Constabulary
5,E23000002,Cumbria,2022,2035.0,2117.0,2151.0,2255.0,2225.0,2413.0,2482.0,...,143280.0,132192.0,127264.0,115868.0,106680.0,517140.0,22574832.0,44.864594,Cumbria Constabulary,Cumbria Constabulary
6,E23000002,Cumbria,2023,1996.0,2083.0,2184.0,2222.0,2338.0,2284.0,2504.0,...,158720.0,135837.0,123574.0,119437.0,109284.0,528955.0,22769056.0,44.934551,Cumbria Constabulary,Cumbria Constabulary
7,E23000002,Cumbria,2024,1901.0,2064.0,2150.0,2267.0,2271.0,2383.0,2349.0,...,167840.0,151470.0,129232.0,116117.0,111048.0,550035.0,22994776.0,45.027759,Cumbria Constabulary,Cumbria Constabulary
8,E23000003,Lancashire,2021,7283.0,7771.0,8142.0,7901.0,8325.0,8688.0,8648.0,...,339280.0,318978.0,307828.0,293156.0,261240.0,1230035.0,63563436.0,41.486215,Lancashire Constabulary,Lancashire Constabulary


In [22]:
pop_cols = [f"F{i}" for i in range(86)] + [f"M{i}" for i in range(86)]


In [23]:
[x for x in pop_cols if x not in pop_df.columns]


[]

In [24]:
pop_df["population_total"] = pop_df[pop_cols].sum(axis=1)


In [25]:
pop_df.head()

,PFA 2023 Code,PFA 2023 Name,Year,F0,F1,F2,F3,F4,F5,F6,...,M80_weighted,M81_weighted,M82_weighted,M83_weighted,M84_weighted,M85_weighted,total_weighted_age,average_age,falls_within_constabulary,force_clean
4,E23000002,Cumbria,2021,2069.0,2089.0,2198.0,2182.0,2384.0,2432.0,2374.0,...,138640.0,134460.0,123738.0,115536.0,105588.0,498440.0,22380241.0,44.695316,Cumbria Constabulary,Cumbria Constabulary
5,E23000002,Cumbria,2022,2035.0,2117.0,2151.0,2255.0,2225.0,2413.0,2482.0,...,143280.0,132192.0,127264.0,115868.0,106680.0,517140.0,22574832.0,44.864594,Cumbria Constabulary,Cumbria Constabulary
6,E23000002,Cumbria,2023,1996.0,2083.0,2184.0,2222.0,2338.0,2284.0,2504.0,...,158720.0,135837.0,123574.0,119437.0,109284.0,528955.0,22769056.0,44.934551,Cumbria Constabulary,Cumbria Constabulary
7,E23000002,Cumbria,2024,1901.0,2064.0,2150.0,2267.0,2271.0,2383.0,2349.0,...,167840.0,151470.0,129232.0,116117.0,111048.0,550035.0,22994776.0,45.027759,Cumbria Constabulary,Cumbria Constabulary
8,E23000003,Lancashire,2021,7283.0,7771.0,8142.0,7901.0,8325.0,8688.0,8648.0,...,339280.0,318978.0,307828.0,293156.0,261240.0,1230035.0,63563436.0,41.486215,Lancashire Constabulary,Lancashire Constabulary


In [26]:
merged_df.head()

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,year,month_num,outcome_type,missing_location,has_outcome,crime_category,urbanisation
0,000004a8bc59a7595f94d35c8b02633eb0dd8077305e28...,2023-07,Lancashire Constabulary,Lancashire Constabulary,-2.503001,53.752564,On or near Leamington Road,E01012588,Blackburn with Darwen 002E,Violence and sexual offences,Status update unavailable,2023,7,NaN,False,False,Violence and sexual offences,Urban Minor Conurbation
1,0000067195b659bd3f0b70409a59cd11816770401ea366...,2025-12,Cambridgeshire Constabulary,Cambridgeshire Constabulary,0.113614,52.173247,On or near Anstey Way,E01035526,Cambridge 015C,Criminal damage and arson,Investigation complete; no suspect identified,2025,12,Investigation complete; no suspect identified,False,True,Damage,Urban City & Town
2,000006a0d6919a710d3bd0a37d0ff31a0eb99ee5a1c20f...,2025-11,Norfolk Constabulary,Norfolk Constabulary,1.213493,52.649220,On or near Chestnut Close,E01026925,South Norfolk 017D,Violence and sexual offences,Unable to prosecute suspect,2025,11,Unable to prosecute suspect,False,True,Violence and sexual offences,Rural Town & Fringe
3,000006d37e5a2ba94f983d908f144b72005d3d4786441e...,2026-01,Hampshire Constabulary,Hampshire Constabulary,-1.578626,50.922591,On or near Barleycorn Walk,E01022978,New Forest 006B,Violence and sexual offences,Under investigation,2026,1,NaN,False,False,Violence and sexual offences,Urban City & Town
4,0000150063e892e027db2aacd0af873ed5ae0c2aba0adb...,2023-09,Avon and Somerset Constabulary,Avon and Somerset Constabulary,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Unable to prosecute suspect,2023,9,Unable to prosecute suspect,True,True,Violence and sexual offences,Urban City & Town


In [27]:
merged_df["year"].unique()

array([2023, 2025, 2026, 2024])

Population data was only available up to 2024. To ensure consistency across crime years (2023–2026), 2024 population estimates were used as a stable denominator for all crime rate calculations. This avoids mismatched denominators and maintains comparability across years.

In [28]:
pop_2024 = pop_df[pop_df["Year"] == 2024].copy()
pop_2024["population_total"] = pop_2024[pop_cols].sum(axis=1)
pop_by_force2 = (
    pop_2024.groupby("force_clean", as_index=False)["population_total"]
    .sum()
)


In [29]:
crime_by_force_year2 = (
    merged_df.groupby(["falls_within", "year"])["crime_id"]
    .count()
    .reset_index(name="crime_count")
)
merged2 = crime_by_force_year2.merge(
    pop_by_force2,
    left_on="falls_within",
    right_on="force_clean",
    how="left"
)
merged2["crime_rate"] = merged2["crime_count"] / merged2["population_total"]
merged2["crime_rate_per_1000"] = merged2["crime_rate"] * 1000


In [30]:
merged2[["crime_rate","force_clean"]]

,crime_rate,force_clean
0,0.073862,Avon and Somerset Constabulary
1,0.092160,Avon and Somerset Constabulary
2,0.094932,Avon and Somerset Constabulary
3,0.007383,Avon and Somerset Constabulary
4,0.071066,Cambridgeshire Constabulary
5,0.074544,Cambridgeshire Constabulary
6,0.075522,Cambridgeshire Constabulary
7,0.006203,Cambridgeshire Constabulary
8,0.063485,Cheshire Constabulary
9,0.067604,Cheshire Constabulary


These calculations create three key demographic groups—working‑age adults, elderly residents, and young people—by summing the relevant age‑specific population columns for each area. The population dataset stores each age as a separate column (e.g., F30, M45), so grouping them into meaningful categories requires manually selecting the appropriate age ranges. Summing ages 16–65 gives the working‑age population, ages 66–85 represent the elderly population, and ages 0–17 represent youth. Creating these aggregated groups is an essential step for analysing how crime patterns relate to the demographic structure of each police force area, and for calculating rates or proportions based on specific age segments.

In [31]:
pop_2024["working_age"] = pop_df[[f"F{i}" for i in range(16, 66)] + [f"M{i}" for i in range(16, 66)]].sum(axis=1)
pop_2024["elderly"] = pop_df[[f"F{i}" for i in range(66, 86)] + [f"M{i}" for i in range(66, 86)]].sum(axis=1)
pop_2024["youth"] = pop_df[[f"F{i}" for i in range(0, 18)] + [f"M{i}" for i in range(0, 18)]].sum(axis=1)
pop_2024.head()

,PFA 2023 Code,PFA 2023 Name,Year,F0,F1,F2,F3,F4,F5,F6,...,M83_weighted,M84_weighted,M85_weighted,total_weighted_age,average_age,falls_within_constabulary,force_clean,working_age,elderly,youth
7,E23000002,Cumbria,2024,1901.0,2064.0,2150.0,2267.0,2271.0,2383.0,2349.0,...,116117.0,111048.0,550035.0,22994776.0,45.027759,Cumbria Constabulary,Cumbria Constabulary,309370.0,120945.0,91642.0
11,E23000003,Lancashire,2024,7342.0,7573.0,8267.0,8130.0,8676.0,8902.0,8694.0,...,285852.0,259392.0,1348695.0,66402724.0,41.459077,Lancashire Constabulary,Lancashire Constabulary,1001495.0,307111.0,332119.0
23,E23000006,Cheshire,2024,4782.0,5132.0,5621.0,5595.0,5826.0,6032.0,6081.0,...,215136.0,204792.0,1066835.0,48609045.0,42.643852,Cheshire Constabulary,Cheshire Constabulary,710040.0,229399.0,227144.0
31,E23000008,Durham,2024,2842.0,2740.0,2929.0,2861.0,3086.0,3045.0,3265.0,...,118939.0,113904.0,556155.0,27688333.0,42.564693,Durham Constabulary,Durham Constabulary,409305.0,132934.0,123203.0
71,E23000018,Derbyshire,2024,4878.0,5211.0,5372.0,5397.0,5490.0,5837.0,5845.0,...,194137.0,193032.0,962200.0,46447367.0,42.358655,Derbyshire Constabulary,Derbyshire Constabulary,687031.0,217217.0,218268.0


In [32]:
pop_by_force2.to_csv("population_2024_by_force.csv", index=False)
crime_by_force_year2.to_csv("crime_by_force_year.csv", index=False)
merged2.to_csv("crime_rates_by_force_year.csv", index=False)
merged_df.to_csv("crime+outcomes.csv", index=False)
pop_2024.to_csv("population_2024_clean.csv",index=False)